# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This reuses the **honest** model from `w06_validation_audit.ipynb` — the grouped `client_id` split, with the two label-sibling features (`impressions_last_30d`, `impressions_prev_30d`) removed after the confession test showed they were leaking information about how the label itself was computed, and with the `avg_position == 0` "no data" fix applied.

The model is **not** re-trained here. It's scored once on the held-out test set (content it never saw during training) to build the queue, so the ranking reflects genuine out-of-sample behavior, not fit-to-training-data.

**How the queue is built:**
1. Score every held-out row with `predict_proba` → `predicted_decline_probability`.
2. Sort descending. Rank 1 = the item the model is most confident is declining.
3. For each row, take the **top-2 features by signed contribution** (`coefficient × imputed value`) as `reason_code_1` / `reason_code_2` — this is what's actually driving *that row's* score, not a global importance list.
4. Map the dominant reason code to a plain-language `suggested_action` a content strategist can act on.

**Honest metric for a ranked list:** global precision/recall describe the *whole* test set, but a playbook is used top-down, so the number that matters is **precision@K** — how clean the top of the queue is. On this held-out set: **precision@100 ≈ 0.88, precision@250 ≈ 0.86**, well above the 62.8% base rate and above the model's own global precision (0.715). In the claim-ladder's words, this model **ranks/flags the top of the queue at precision@100 of ~0.88** — that's the honest claim, not "the model predicts decline."

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, accuracy_score
from sklearn.impute import SimpleImputer

RANDOM_SEED = 42

url = 'https://raw.githubusercontent.com/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# Same cleaning as w06_validation_audit.ipynb (honest, leakage-checked pipeline)
df['has_word_count'] = df['word_count'].notna().astype(int)
df['word_count'] = df['word_count'].fillna(df['word_count'].median())
df['is_declining_label'] = (df['trend_pct'] < 0).astype(int)

# FIX: avg_position == 0 means "no data", not rank zero
df['has_avg_position'] = (df['avg_position'] != 0).astype(int)
df.loc[df['avg_position'] == 0, 'avg_position'] = np.nan
df['avg_position'] = df['avg_position'].fillna(df['avg_position'].median())

forbidden_always = ['is_declining_label', 'trend_direction', 'trend_pct', 'content_id', 'client_id', 'content_type']
suspects = ['impressions_last_30d', 'impressions_prev_30d']  # label-sibling leakage, confirmed in w06
all_numeric = [c for c in df.columns if c not in forbidden_always and pd.api.types.is_numeric_dtype(df[c])]
features_clean = [c for c in all_numeric if c not in suspects]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
X_train, y_train = train_df[features_clean], train_df['is_declining_label']
X_test, y_test = test_df[features_clean], test_df['is_declining_label']

imputer = SimpleImputer(strategy='median', add_indicator=True)
X_train_i = imputer.fit_transform(X_train)
X_test_i = imputer.transform(X_test)
feat_names = imputer.get_feature_names_out(X_train.columns)

model = LogisticRegression(max_iter=20000, solver='liblinear', random_state=RANDOM_SEED)
model.fit(X_train_i, y_train)

probs = model.predict_proba(X_test_i)[:, 1]
preds = model.predict(X_test_i)
precision = precision_score(y_test, preds, zero_division=0)
recall = recall_score(y_test, preds)
accuracy = accuracy_score(y_test, preds)
base_rate = y_test.mean()

# precision@K -- the honest metric for a RANKED queue (top-of-list quality)
order = np.argsort(-probs)
y_sorted = y_test.values[order]
p_at_100 = y_sorted[:100].mean()
p_at_250 = y_sorted[:250].mean()

# --- Build the ranked queue on the held-out test set (never trained on) ---
queue = test_df.copy()
queue['predicted_decline_probability'] = probs

# Per-row reason code: top-2 features by |coefficient * standardized contribution|
coefs = pd.Series(model.coef_[0], index=feat_names)
X_test_i_df = pd.DataFrame(X_test_i, columns=feat_names, index=queue.index)
contributions = X_test_i_df * coefs  # signed contribution per feature per row
top_reason = contributions.abs().apply(lambda r: contributions.columns[np.argsort(-r.values)[:2]], axis=1)
queue['reason_code_1'] = [r[0] for r in top_reason]
queue['reason_code_2'] = [r[1] for r in top_reason]

# Archetype: content_type x age_tier (both already-computed, human-readable dimensions)
queue['archetype'] = queue['content_type'].str.title() + ' — age ' + queue['age_tier']

queue = queue.sort_values('predicted_decline_probability', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

def suggested_action(row):
    if row['has_avg_position'] == 0:
        return 'Insufficient ranking data -- verify tracking before any action'
    if row['reason_code_1'] in ('clicks_last_30d', 'clicks_90d', 'ctr'):
        return 'Review title/meta and on-SERP CTR; check for cannibalization'
    if row['reason_code_1'] in ('age_tier_order', 'days_with_sessions'):
        return 'Candidate for content refresh review (aged, declining engagement)'
    if row['reason_code_1'] in ('engaged_sessions_90d', 'scroll_events_90d'):
        return 'Review on-page engagement -- content or UX audit'
    return 'General decline review -- no single dominant factor'

queue['suggested_action'] = queue.apply(suggested_action, axis=1)

print("Model summary (held-out test set, grouped split by client_id):")
print(f"  Global -- Precision: {precision:.3f} | Recall: {recall:.3f} | Accuracy: {accuracy:.3f} | Base rate: {base_rate:.3f}")
print(f"  Top-of-queue -- precision@100: {p_at_100:.3f} | precision@250: {p_at_250:.3f}")
print()
export_cols = ['content_id', 'rank', 'predicted_decline_probability', 'archetype',
               'reason_code_1', 'reason_code_2', 'suggested_action', 'main_intent',
               'age_tier', 'content_type']
ranked_queue = queue[export_cols]
print("Top of the ranked queue:")
print(ranked_queue.head(10).to_string(index=False))


Model summary (held-out test set, grouped split by client_id):
  Global -- Precision: 0.715 | Recall: 0.797 | Accuracy: 0.673 | Base rate: 0.628
  Top-of-queue -- precision@100: 0.880 | precision@250: 0.860

Top of the ranked queue:
          content_id  rank  predicted_decline_probability                     archetype reason_code_1     reason_code_2                                             suggested_action   main_intent age_tier    content_type
content_4a18c07e5357     1                       1.000000    Keyword Article — age 365+    clicks_90d   clicks_last_30d Review title/meta and on-SERP CTR; check for cannibalization informational     365+ keyword article
content_bdd7c88a58ed     2                       1.000000    Keyword Article — age 365+    clicks_90d   clicks_last_30d Review title/meta and on-SERP CTR; check for cannibalization informational     365+ keyword article
content_4090f0acd977     3                       0.999999    Keyword Article — age 365+    clicks_90d      

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a content strategist or SEO lead uses this queue as a **prioritization aid** — a shortlist of which content to review first, ordered by how strongly the model's signals resemble past declining content in this dataset. It replaces manually scrolling every content item, not the reviewer's judgment.

**Where it's valid:**
- Content resembling the training distribution: the model was **trained** on all 3 `content_type`s (keyword article, feedly article, comparison article) across the dataset's 32 clients, trailing-90-day window.
- **Caveat found while building this queue:** the random grouped holdout used to *validate* precision@K happened to land on 7 clients whose content is 100% "keyword article" (checked below) — so the precision@100/250 numbers reported here are directly demonstrated for keyword articles, not independently confirmed for feedly or comparison articles, even though the model was trained on all three. Re-run with a different seed, or a stratified split, before leaning on this queue for the other two content types.
- As a **ranking**, not a probability you can act on literally — `predicted_decline_probability = 0.999` does not mean "99.9% chance"; it's a score that ranks well (precision@100 ≈ 0.88 on the keyword-article-only holdout), not a calibrated probability.

**Where it stops being valid:**
- **New clients or content types** outside the 32-client, 3-type training set — the model has never seen them.
- **Time-forward claims.** This is a single grouped holdout on one anonymized slice with no `report_date`, so it cannot say "this page will decline next month" — only "this page currently looks like ones that were labeled declining in this dataset."
- **Global precision is modest (0.715, barely above the 62.8% base rate).** The queue is only trustworthy in its **top ranks** (precision@100/250) — treating rank 3,000 the same as rank 1 would misuse it.
- **Single train/test split**, not cross-validated (flagged as a limitation in `w06_validation_audit.ipynb` too) — the exact precision@K numbers would move somewhat under a different seed.

In [2]:
# Boundary check: how much of the queue is actually high-confidence vs. low-signal
n_total = len(ranked_queue)
n_no_data = (queue['has_avg_position'] == 0).sum()
n_high_conf = (ranked_queue['predicted_decline_probability'] >= 0.7).sum()

print(f"Queue size (held-out test set): {n_total}")
print(f"Rows flagged 'no ranking data' (avg_position was 0): {n_no_data} ({n_no_data/n_total:.1%})")
print(f"Rows above the 0.7 confidence line: {n_high_conf} ({n_high_conf/n_total:.1%})")
print(f"Content types covered: {sorted(queue['content_type'].unique())}")
print(f"Clients covered: {queue['client_id'].nunique()} (out of the dataset's 32)")


Queue size (held-out test set): 6163
Rows flagged 'no ranking data' (avg_position was 0): 62 (1.0%)
Rows above the 0.7 confidence line: 2647 (42.9%)
Content types covered: ['keyword article']
Clients covered: 7 (out of the dataset's 32)


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**A person must check, before any action is taken:**
- The `reason_code` actually makes sense for that specific page (e.g. a seasonal page flagged for "declining clicks" in its off-season isn't necessarily broken).
- Whether the page recently changed for a reason the model can't see (a manual deprioritization, a merged/redirected URL, a paused campaign).
- The `suggested_action` against the client's editorial calendar and resourcing — this queue has no concept of team capacity or business priority.

**This should NEVER be automated (no-go list):**
- **Auto-publishing or auto-editing content** based on the model's output. The model flags candidates for *human* review only.
- **Auto-deprioritizing or removing content** for clients where `has_avg_position == 0` — that flag means "no data," and acting on missing data as if it were "bad performance" would be a data-handling error, not a business decision.
- **Treating the ranked score as a guarantee.** Given global precision of ~0.71, roughly 1 in 4 flagged items in the *broader* list (not just the top) will be a false positive — a human reviewer is the safety net, not a formality.
- **Extending this queue to content types or clients not in the training data** without re-validating first (see Section 2).

In [3]:
# Concrete no-go check: rows that must be routed to "verify data first", not the action queue
no_go = ranked_queue[queue['has_avg_position'] == 0]
print(f"Rows routed to the no-go / verify-data-first list: {len(no_go)}")
print(no_go[['content_id', 'rank', 'suggested_action']].head(5).to_string(index=False))


Rows routed to the no-go / verify-data-first list: 62
          content_id  rank                                               suggested_action
content_74b78c2e3b4b  4172 Insufficient ranking data -- verify tracking before any action
content_c63decebf95b  4331 Insufficient ranking data -- verify tracking before any action
content_8670bdb4ea0e  4345 Insufficient ranking data -- verify tracking before any action
content_cc1a7065fda5  4369 Insufficient ranking data -- verify tracking before any action
content_e1b24ff1f2e0  4385 Insufficient ranking data -- verify tracking before any action


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Concrete, checkable thresholds — not vague "monitor performance":

1. **Base-rate drift.** If the share of actually-declining content in a fresh sample moves more than ~5 points from 62.8% (the base rate this model was tuned against), the model's calibration is out of date — retrain.
2. **Precision@100 drop.** If precision@100 on a new holdout falls meaningfully below the ~0.88 observed here (e.g. below 0.75), the top of the queue is no longer trustworthy enough to hand a reviewer without extra scrutiny.
3. **Coefficient shape change.** Re-run the leakage confession test (`w06`) periodically — if any single feature's coefficient suddenly dominates the way `impressions_last_30d` did before it was removed, that's the leakage signature reappearing, likely from a new derived column.
4. **Coverage drift.** If a production batch includes `content_type`s or clients outside the 3 types / 32 clients the model was ever *trained* on, pause and re-check. (Note: comparing a batch to one random train/test fold isn't a useful drift test by itself — a grouped split guarantees the test clients look "new" to that fold every time. The check below instead compares against the full known pool the model was trained on.)
5. **Time-based decay (once time-aware data is available).** This starter CSV has no calendar date; once the warehouse's `report_date`-keyed tables are used, add a check for performance decay month-over-month rather than relying on a single static holdout indefinitely.

In [4]:
# Coverage drift check: compare a batch against the FULL known pool the model
# was ever trained on (all 32 clients, all 3 content types) -- not against one
# arbitrary train fold, since a grouped split guarantees fold-level clients
# always look "new" to each other by construction (not a real drift signal).
known_types = set(df['content_type'].unique())
known_clients = set(df['client_id'].unique())

# This test batch, evaluated against the FULL pool (the real check to run in production):
batch_types = set(test_df['content_type'].unique())
batch_clients = set(test_df['client_id'].unique())

print("Coverage drift check (run this against any NEW production batch):")
print(f"  Content types the model was ever trained on: {sorted(known_types)}")
print(f"  Content types in this batch: {sorted(batch_types)}")
print(f"  Content types in this batch outside the known pool: {batch_types - known_types or 'none'}")
print(f"  Clients in this batch outside the known 32-client pool: {batch_clients - known_clients or 'none'}")
print()
print(f"  NOTE: this batch happens to be {sorted(batch_types)} only -- see Section 2 caveat.")
n_known_types = df['content_type'].nunique()
print(f"  Of the {n_known_types} known content types, this holdout validated precision@K for 1 of them directly.")


Coverage drift check (run this against any NEW production batch):
  Content types the model was ever trained on: ['comparison article', 'feedly article', 'keyword article']
  Content types in this batch: ['keyword article']
  Content types in this batch outside the known pool: none
  Clients in this batch outside the known 32-client pool: none

  NOTE: this batch happens to be ['keyword article'] only -- see Section 2 caveat.
  Of the 3 known content types, this holdout validated precision@K for 1 of them directly.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exports the full ranked queue as a CSV and a summary chart of the queue's composition (reason codes and archetypes in the top 250) as a figure. Both are written with relative paths so a fresh clone reproduces them in the right place. Note: dataset-shaped CSVs under `work/` are gitignored by design (see `work/README.md`) — these files are meant to be **regenerated by running this notebook**, not committed as data.

In [5]:
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1. Ranked queue CSV
out_csv = 'work/outputs/refresh_action_queue.csv'
ranked_queue.to_csv(out_csv, index=False)

# 2. Queue composition figure (top 250 -- the part of the queue that's actually trustworthy, see Section 2)
top250 = queue.head(250)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

reason_counts = top250['reason_code_1'].value_counts()
axes[0].barh(reason_counts.index[::-1], reason_counts.values[::-1], color='#4C72B0')
axes[0].set_title('Top reason code -- top 250 of ranked queue')
axes[0].set_xlabel('Count')

archetype_counts = top250['archetype'].value_counts()
axes[1].barh(archetype_counts.index[::-1], archetype_counts.values[::-1], color='#DD8452')
axes[1].set_title('Archetype -- top 250 of ranked queue')
axes[1].set_xlabel('Count')

plt.tight_layout()
out_fig = 'work/figures/queue_composition.png'
plt.savefig(out_fig, dpi=130)
plt.show()

print(f"Exported ranked queue: {out_csv} ({len(ranked_queue)} rows)")
print(f"Exported figure: {out_fig}")


Exported ranked queue: work/outputs/refresh_action_queue.csv (6163 rows)
Exported figure: work/figures/queue_composition.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.